In [1]:
!pip install transformers torch spacy scikit-learn jsonlines
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 75.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import json
import jsonlines
import torch
import spacy
import numpy as np

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_score, recall_score, f1_score

Load SciBERT

In [3]:
MODEL_NAME = "allenai/scibert_scivocab_cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31116, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

Load SpaCy (for candidate entities)

In [4]:
nlp = spacy.load("en_core_web_sm")

Define Labels

In [5]:
labels = {
    "TOPIC": "main subject or research topic",
    "METHOD": "method or technique used to perform a task",
    "CONCEPT": "general concept or idea in computer science",
    "ALGORITHM": "step by step computational algorithm",
    "OTHER": "other technical term"
}

Embedding Function (SciBERT)

In [6]:
def embed(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    emb = outputs.last_hidden_state.mean(dim=1)

    return emb.cpu().numpy()

Create Label Embeddings

In [7]:
label_embeddings = {}

for label, desc in labels.items():
    label_embeddings[label] = embed(desc)

Candidate Entity Extraction

In [8]:
def extract_candidates(text):

    doc = nlp(text)

    candidates = set()

    for chunk in doc.noun_chunks:
        candidates.add(chunk.text.strip())

    return list(candidates)

Label Prediction

In [9]:
def predict_entity_label(entity):

    e_vec = embed(entity)

    best_label = None
    best_score = -1

    for label in labels:

        score = cosine_similarity(
            e_vec,
            label_embeddings[label]
        )[0][0]

        if score > best_score:
            best_score = score
            best_label = label

    return best_label

Predict Entities for One Sample

In [10]:
def predict_entities(text):

    candidates = extract_candidates(text)

    predictions = []

    for c in candidates:

        label = predict_entity_label(c)

        predictions.append({
            "entity": c,
            "label": label
        })

    return predictions

In [11]:
def compute_accuracy(y_true, y_pred):

    correct = 0

    for t, p in zip(y_true, y_pred):
        if t == p:
            correct += 1

    return correct / len(y_true)

def compute_partial_accuracy(gold_entities, pred_entities):

    match = 0

    for g in gold_entities:

        g_text = g["entity"].lower()

        for p in pred_entities:

            p_text = p["entity"].lower()

            if g_text in p_text or p_text in g_text:
                match += 1
                break

    return match / len(gold_entities)

Load Dataset

In [12]:
data = []

with jsonlines.open("OS-cleaned_file.jsonl") as reader:
    for obj in reader:
        data.append(obj)

Evaluation

In [13]:
import json
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = []
y_pred = []

partial_scores = []

for sample in data:

    text = sample["input"]

    gold = json.loads(sample["output"])

    pred = predict_entities(text)

    gold_dict = {g["entity"]: g["label"] for g in gold}
    pred_dict = {p["entity"]: p["label"] for p in pred}

    # label comparison
    for ent in gold_dict:

        y_true.append(gold_dict[ent])

        if ent in pred_dict:
            y_pred.append(pred_dict[ent])
        else:
            y_pred.append("OTHER")

    # partial accuracy
    p_score = compute_partial_accuracy(gold, pred)
    partial_scores.append(p_score)

Metrics

In [14]:
# MICRO metrics
micro_precision = precision_score(y_true, y_pred, average="micro")
micro_recall = recall_score(y_true, y_pred, average="micro")
micro_f1 = f1_score(y_true, y_pred, average="micro")

# MACRO metrics
macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")
macro_f1 = f1_score(y_true, y_pred, average="macro")

# Accuracy
accuracy = compute_accuracy(y_true, y_pred)

# Partial Accuracy
partial_accuracy = sum(partial_scores) / len(partial_scores)

print("\n===== Evaluation Results =====")

print("Accuracy:", round(accuracy,4))
print("Partial Accuracy:", round(partial_accuracy,4))

print("\nMicro Precision:", round(micro_precision,4))
print("Micro Recall:", round(micro_recall,4))
print("Micro F1:", round(micro_f1,4))

print("\nMacro Precision:", round(macro_precision,4))
print("Macro Recall:", round(macro_recall,4))
print("Macro F1:", round(macro_f1,4))


===== Evaluation Results =====
Accuracy: 0.1519
Partial Accuracy: 0.9335

Micro Precision: 0.1519
Micro Recall: 0.1519
Micro F1: 0.1519

Macro Precision: 0.0902
Macro Recall: 0.1026
Macro F1: 0.0306


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Example Output

In [15]:
sample = data[0]

print("TEXT:\n", sample["input"])

print("\nPREDICTED:")
print(predict_entities(sample["input"]))

print("\nGROUND TRUTH:")
print(sample["output"])

TEXT:
 Context Switching Context switching is a fundamental mechanism that enables multitasking in modern operating systems. When the CPU switches from executing one process to another, the operating system must save the complete execution context of the current process, including the program counter, CPU registers, and memory management information, into its Process Control Block (PCB). The context of the next selected process is then loaded so execution can resume seamlessly. This mechanism allows multiple processes to share a single CPU by rapidly alternating execution, creating the illusion of parallelism. However, context switching introduces overhead because no useful work is done while the switch is occurring. The time spent saving and restoring contexts depends on hardware support and operating system design. Frequent context switches can significantly degrade system performance, especially in systems with small time quanta. Therefore, OS designers strive to minimize context sw